In [ ]:
%pip install azure.ai.documentintelligence
%pip install dotenv

In [2]:
import os

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult
from dotenv import load_dotenv

In [7]:
def _format_price(price_dict):
    if price_dict is None:
        return "N/A"
    return "".join([f"{p}" for p in price_dict.values()])

In [3]:
load_dotenv()
key      = os.environ["key"]
endpoint = os.environ["endpoint"]

In [4]:
document_intelligence_client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

In [5]:
path = os.path.join('receipt.png')

In [6]:
with open(path, "rb") as f:
    poller = document_intelligence_client.begin_analyze_document("prebuilt-receipt", body=f, locale="en-US")
receipts: AnalyzeResult = poller.result()

In [8]:
if receipts.documents:
    for idx, receipt in enumerate(receipts.documents):
        print(f"--------Analysis of receipt #{idx + 1}--------")
        print(f"Receipt type: {receipt.doc_type if receipt.doc_type else 'N/A'}")
        if receipt.fields:
            merchant_name = receipt.fields.get("MerchantName")
            if merchant_name:
                print(
                    f"Merchant Name: {merchant_name.get('valueString')} has confidence: "
                    f"{merchant_name.confidence}"
                )
            transaction_date = receipt.fields.get("TransactionDate")
            if transaction_date:
                print(
                    f"Transaction Date: {transaction_date.get('valueDate')} has confidence: "
                    f"{transaction_date.confidence}"
                )
            items = receipt.fields.get("Items")
            if items:
                print("Receipt items:")
                for idx, item in enumerate(items.get("valueArray")):
                    print(f"...Item #{idx + 1}")
                    item_description = item.get("valueObject").get("Description")
                    if item_description:
                        print(
                            f"......Item Description: {item_description.get('valueString')} has confidence: "
                            f"{item_description.confidence}"
                        )
                    item_quantity = item.get("valueObject").get("Quantity")
                    if item_quantity:
                        print(
                            f"......Item Quantity: {item_quantity.get('valueString')} has confidence: "
                            f"{item_quantity.confidence}"
                        )
                    item_total_price = item.get("valueObject").get("TotalPrice")
                    if item_total_price:
                        print(
                            f"......Total Item Price: {_format_price(item_total_price.get('valueCurrency'))} has confidence: "
                            f"{item_total_price.confidence}"
                        )
            subtotal = receipt.fields.get("Subtotal")
            if subtotal:
                print(
                    f"Subtotal: {_format_price(subtotal.get('valueCurrency'))} has confidence: {subtotal.confidence}"
                )
            tax = receipt.fields.get("TotalTax")
            if tax:
                print(f"Total tax: {_format_price(tax.get('valueCurrency'))} has confidence: {tax.confidence}")
            tip = receipt.fields.get("Tip")
            if tip:
                print(f"Tip: {_format_price(tip.get('valueCurrency'))} has confidence: {tip.confidence}")
            total = receipt.fields.get("Total")
            if total:
                print(f"Total: {_format_price(total.get('valueCurrency'))} has confidence: {total.confidence}")
        print("--------------------------------------")

--------Analysis of receipt #1--------
Receipt type: receipt.retailMeal
Merchant Name: Alamo Drafthouse Cinema has confidence: 0.978
Transaction Date: 2009-05-07 has confidence: 0.988
Receipt items:
...Item #1
......Item Description: SM Water has confidence: 0.982
......Total Item Price: 0.0USD has confidence: 0.988
...Item #2
......Item Description: earl gray tea has confidence: 0.98
......Total Item Price: 2.5USD has confidence: 0.988
...Item #3
......Item Description: Romulan Ale has confidence: 0.982
......Total Item Price: 5.5USD has confidence: 0.988
...Item #4
......Item Description: klingon wine has confidence: 0.982
......Total Item Price: 5.5USD has confidence: 0.988
...Item #5
......Item Description: Royale w/chz has confidence: 0.983
......Total Item Price: 8.99USD has confidence: 0.988
Subtotal: 22.49USD has confidence: 0.983
Total tax: 1.86USD has confidence: 0.988
Total: 24.35USD has confidence: 0.987
--------------------------------------
